### Ingest constructor.json file


In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "Ergast")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2025-01")
v_file_date = dbutils.widgets.get("p_file_date")
raw_race_path = f"{raw_folder_path}/{v_file_date}"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

In [0]:
constructors_schema_incremental = StructType([
  StructField("constructorId", StringType(), False),
  StructField("name", StringType(), True),
  StructField("nationality", StringType(), True),
  StructField("url", StringType(), True)
])

constructor_schema_static = StructType([
  StructField("constructorId", IntegerType(), False),
  StructField("constructorRef", StringType(), True),
  StructField("name", StringType(), True),
  StructField("nationality", StringType(), True),
  StructField("url", StringType(), True)
])

if USE_INCREMENTAL:
  constructor_schema = constructors_schema_incremental
else:
  constructor_schema = constructor_schema_static

constructor_df = spark.read.json(
 f"{raw_race_path}/constructors.json", 
  schema=constructor_schema
  )
display(constructor_df)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
if USE_INCREMENTAL:
    final_constructor_df = (
        constructor_df
        .withColumnRenamed("constructorId", "constructor_id")
        .drop("url")
        .withColumn("data_source", lit(v_data_source))
        .withColumn("file_date", lit(v_file_date))
    )
    final_constructor_df = add_ingestion_date(final_constructor_df)
else:
    renamed_constructor_df = (
        constructor_df
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("constructorRef", "constructor_ref")
    )
    selected_constructor_df = renamed_constructor_df.drop("url")
    final_constructor_df = add_ingestion_date(selected_constructor_df)

display(final_constructor_df)


In [0]:
final_constructor_df.write.mode("overwrite").format("delta").save(f"{processed_folder_path}/constructors")

In [0]:
df = spark.read.format("delta").load(f"{processed_folder_path}/constructors")
display(df)

In [0]:
dbutils.notebook.exit("Success")